In [53]:
import pickle
from timeit import default_timer as timer
import os
import random

import numpy as np
from sklearn.decomposition import PCA
from faerun import Faerun

def main(fold_len=2048, mit_limit=100000):
    """
    Load DRFP fingerprint pickles for USPTO-MIT and Chan–Lam,
    compute a 2D PCA embedding and create an interactive Faerun plot.
    """

    # Build paths for pickled fingerprints (created by earlier cells)
    mit_path = f"fingerprints/drfp_fps_mit_{fold_len}.pkl"
    chanlam_path = f"fingerprints/drfp_fps_chanlam_{fold_len}.pkl"

    if not os.path.exists(mit_path) or not os.path.exists(chanlam_path):
        print("Missing fingerprint files:", mit_path, chanlam_path)
        return

    # Load and sample MIT (shuffle for a fair subset)
    with open(mit_path, "rb") as f:
        fps_mit_full = pickle.load(f)
    fps_mit = fps_mit_full.copy()
    random.shuffle(fps_mit)
    fps_mit = fps_mit[:mit_limit]

    # Load Chan–Lam (use all available)
    with open(chanlam_path, "rb") as f:
        fps_chanlam = pickle.load(f)

    # Convert to boolean arrays and stack
    fps_mit = np.array(fps_mit, dtype=bool)
    fps_chanlam = np.array(fps_chanlam, dtype=bool)
    fps_combined = np.vstack([fps_mit, fps_chanlam])

    print(f"Combined fingerprints shape: {fps_combined.shape}")

    # Compute a fast 2D embedding using PCA
    start = timer()
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(fps_combined.astype(float))
    end = timer()
    print(f"PCA completed in {end - start:.2f}s. Explained var: {pca.explained_variance_ratio_}")

    # Build labels and color indices (0 = MIT, 1 = Chan–Lam)
    n_mit = len(fps_mit)
    n_chan = len(fps_chanlam)
    labels = [f"USPTO-MIT__{i}" for i in range(n_mit)] + [f"Chan-Lam__{i}" for i in range(n_chan)]
    colors_idx = [0] * n_mit + [1] * n_chan
    legend_labels = [(0, "USPTO-MIT"), (1, "Chan-Lam")]

    # Create Faerun plot (interactive HTML)
    faerun = Faerun(view="front", coords=False)
    faerun.add_scatter(
        "drfp_pca",
        {"x": coords[:, 0], "y": coords[:, 1], "c": colors_idx, "labels": labels},
        colormap="tab10",
        categorical=True,
        point_scale=2.0,
        has_legend=True,
        shader="sphere",
        legend_labels=legend_labels,
        title_index=1,
    )

    output_file = f"fingerprint_viz_faerun_{fold_len}"
    print("Saving Faerun visualization to:", output_file)
    faerun.plot(output_file)
    print("Done. Open the generated HTML in a browser to explore.")

if __name__ == "__main__":
    main()

Combined fingerprints shape: (109602, 2048)
PCA completed in 5.11s. Explained var: [0.05972856 0.02441014]
Saving Faerun visualization to: fingerprint_viz_faerun_2048
PCA completed in 5.11s. Explained var: [0.05972856 0.02441014]
Saving Faerun visualization to: fingerprint_viz_faerun_2048


c:\Users\Happy\SRP\SRP Project MT training\mt_training\fingerprint_viz_faerun_2048.html

Done. Open the generated HTML in a browser to explore.


In [38]:
import pickle
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import plotly.express as px
import os

In [39]:
# Load pickled fingerprints for all folding lengths
# Shuffle and limit MIT to 10k rows for fair sampling
import random

folding_lengths = [512, 1024, 2048]
mit_limit = 10000
fps_dict = {}

for fold_len in folding_lengths:
    try:
        with open(f"fingerprints/drfp_fps_mit_{fold_len}.pkl", "rb") as f:
            fps_mit_full = pickle.load(f)
            
            # Shuffle the full list
            fps_mit_shuffled = fps_mit_full.copy()
            random.shuffle(fps_mit_shuffled)
            
            # Take first mit_limit from shuffled list
            fps_mit = fps_mit_shuffled[:mit_limit]
        
        with open(f"fingerprints/drfp_fps_chanlam_{fold_len}.pkl", "rb") as f:
            fps_chanlam = pickle.load(f)
        
        fps_dict[fold_len] = {"mit": fps_mit, "chanlam": fps_chanlam}
        print(f"✓ Loaded {fold_len}: {len(fps_mit)} MIT (shuffled from {len(fps_mit_full)}) + {len(fps_chanlam)} Chan-Lam fingerprints")
    except FileNotFoundError as e:
        print(f"✗ Missing fingerprints for {fold_len}: {e}")

print(f"\nTotal folding lengths loaded: {len(fps_dict)}")


✓ Loaded 512: 10000 MIT (shuffled from 479035) + 9602 Chan-Lam fingerprints
✓ Loaded 1024: 10000 MIT (shuffled from 479035) + 9602 Chan-Lam fingerprints
✓ Loaded 1024: 10000 MIT (shuffled from 479035) + 9602 Chan-Lam fingerprints
✓ Loaded 2048: 10000 MIT (shuffled from 479035) + 9602 Chan-Lam fingerprints

Total folding lengths loaded: 3
✓ Loaded 2048: 10000 MIT (shuffled from 479035) + 9602 Chan-Lam fingerprints

Total folding lengths loaded: 3


In [40]:
# Helper function to compute pairwise distances from binary fingerprints (memory-efficient)
def compute_distances(fps, metric="jaccard", batch_size=1000):
    """
    Compute pairwise distances from binary fingerprints (memory-efficient version).
    Uses batched computation to avoid allocating huge matrices.
    
    metric: "jaccard" or "tanimoto" (equivalent for bit vectors)
    batch_size: Process in chunks to save memory
    """
    fps_array = np.array(fps, dtype=bool)
    n = len(fps_array)
    
    # Initialize distance matrix
    distances = np.zeros((n, n), dtype=np.float32)
    
    # Compute in batches to avoid memory overflow
    for i in range(0, n, batch_size):
        end_i = min(i + batch_size, n)
        batch_i = fps_array[i:end_i]
        
        for j in range(0, n, batch_size):
            end_j = min(j + batch_size, n)
            batch_j = fps_array[j:end_j]
            
            # Compute pairwise Tanimoto for this batch
            intersection = np.sum(batch_i[:, np.newaxis, :] & batch_j[np.newaxis, :, :], axis=2)
            union = np.sum(batch_i[:, np.newaxis, :] | batch_j[np.newaxis, :, :], axis=2)
            
            # Avoid division by zero
            with np.errstate(divide='ignore', invalid='ignore'):
                tanimoto_similarity = intersection / union
                tanimoto_similarity = np.nan_to_num(tanimoto_similarity, nan=1.0)
            
            tanimoto_distance = 1.0 - tanimoto_similarity
            distances[i:end_i, j:end_j] = tanimoto_distance
    
    return distances


In [49]:
# Create interactive visualizations using Faerun
# Faerun provides beautiful, interactive 2D/3D visualizations of fingerprint spaces

from faerun import Faerun
import matplotlib
from matplotlib import colors as mcolors

def create_faerun_visualization(fps_mit, fps_chanlam, fold_len, output_name="fingerprint_viz"):
    """
    Create an interactive Faerun visualization of fingerprint embeddings.
    
    Args:
        fps_mit: Binary fingerprints for USPTO-MIT dataset
        fps_chanlam: Binary fingerprints for Chan-Lam dataset
        fold_len: Folding length (for display purposes)
        output_name: Name for the output HTML file
    """
    print(f"\n{'='*60}")
    print(f"Creating Faerun Visualization for Folding Length: {fold_len}")
    print(f"{'='*60}")
    
    # Convert to binary arrays
    fps_mit = np.array(fps_mit, dtype=bool)
    fps_chanlam = np.array(fps_chanlam, dtype=bool)
    fps_combined = np.vstack([fps_mit, fps_chanlam])
    
    print(f"Computing PCA embedding ({len(fps_combined)} fingerprints)...")
    
    # Use PCA for initial embedding (fast)
    embedder = PCA(n_components=2, random_state=42)
    coords = embedder.fit_transform(fps_combined.astype(float))
    
    print(f"✓ Embedding computed")
    print(f"  Explained variance ratio: {embedder.explained_variance_ratio_}")
    
    # Create labels and colors
    n_mit = len(fps_mit)
    dataset_labels = ["USPTO-MIT"] * n_mit + ["Chan-Lam"] * len(fps_chanlam)
    
    # Map labels to integer colors (0 for MIT, 1 for Chan-Lam)
    colors_idx = [0 if label == "USPTO-MIT" else 1 for label in dataset_labels]
    
    # Build a ListedColormap from hex colors so Faerun can use it
    cmap = mcolors.ListedColormap(["#1f77b4", "#ff7f0e"])  # Blue, Orange
    
    print(f"Creating Faerun plot...")
    
    # Initialize Faerun with coords
    faerun = Faerun(view="default", coords=coords)
    
    # Add scatter plot with colors based on dataset
    faerun.add_scatter(
        "DRFP Fingerprints",
        {
            "x": coords[:, 0],
            "y": coords[:, 1],
            "c": colors_idx,
            "labels": dataset_labels
        },
        colormap=cmap,
        categorical=True,
        point_scale=4.0,
        has_legend=True,
        series_title=f"DRFP Fingerprints (Folding Length: {fold_len})"
    )
    
    output_file = f"{output_name}_faerun_{fold_len}.html"
    print(f"Saving to {output_file}...")
    faerun.plot(output_file)
    
    print(f"✓ Visualization saved: {output_file}")
    return coords, dataset_labels


In [ ]:
# Run a single folding length with kNN overlay (k=3)
# This cell will call the visualization function and save an HTML file
fold_len = 2048
# Ensure fps_dict is available (earlier cells may have populated it)
try:
    fps_mit = fps_dict[fold_len]['mit']
    fps_chanlam = fps_dict[fold_len]['chanlam']
except Exception as e:
    print('fps_dict missing or not populated, loading pickles directly...')
    with open(f"fingerprints/drfp_fps_mit_{fold_len}.pkl", 'rb') as f:
        fps_mit = pickle.load(f)
    with open(f"fingerprints/drfp_fps_chanlam_{fold_len}.pkl", 'rb') as f:
        fps_chanlam = pickle.load(f)

# Run visualization with kNN tree overlay
coords, labels = create_faerun_visualization(fps_mit, fps_chanlam, fold_len, output_name="fingerprint_viz", tree_mode='knn', knn_k=3)
print('Visualization with kNN overlay complete.')

In [50]:
# Generate visualizations for all folding lengths using Faerun
# Creates beautiful interactive HTML visualizations

embeddings = {}
labels_dict = {}

for fold_len in folding_lengths:
    if fold_len not in fps_dict:
        print(f"Skipping {fold_len}: fingerprints not loaded")
        continue
    
    fps_mit = fps_dict[fold_len]["mit"]
    fps_chanlam = fps_dict[fold_len]["chanlam"]
    
    print(f"\n>>> Processing folding length {fold_len}")
    
    # Create Faerun visualization
    output_name = f"fingerprint_viz"
    coords, labels = create_faerun_visualization(
        fps_mit, fps_chanlam, fold_len, output_name=output_name
    )
    
    embeddings[fold_len] = coords
    labels_dict[fold_len] = labels



>>> Processing folding length 512

Creating Faerun Visualization for Folding Length: 512
Computing PCA embedding (19602 fingerprints)...
✓ Embedding computed
  Explained variance ratio: [0.172826   0.04952973]
Creating Faerun plot...
Saving to fingerprint_viz_faerun_512.html...


c:\Users\Happy\SRP\SRP Project MT training\mt_training\fingerprint_viz_faerun_512.html.html

✓ Visualization saved: fingerprint_viz_faerun_512.html

>>> Processing folding length 1024

Creating Faerun Visualization for Folding Length: 1024
Computing PCA embedding (19602 fingerprints)...
✓ Embedding computed
  Explained variance ratio: [0.17354668 0.0495672 ]
Creating Faerun plot...
Saving to fingerprint_viz_faerun_1024.html...
✓ Embedding computed
  Explained variance ratio: [0.17354668 0.0495672 ]
Creating Faerun plot...
Saving to fingerprint_viz_faerun_1024.html...


c:\Users\Happy\SRP\SRP Project MT training\mt_training\fingerprint_viz_faerun_1024.html.html

✓ Visualization saved: fingerprint_viz_faerun_1024.html

>>> Processing folding length 2048

Creating Faerun Visualization for Folding Length: 2048
Computing PCA embedding (19602 fingerprints)...
✓ Embedding computed
  Explained variance ratio: [0.17540874 0.04705278]
Creating Faerun plot...
Saving to fingerprint_viz_faerun_2048.html...
✓ Embedding computed
  Explained variance ratio: [0.17540874 0.04705278]
Creating Faerun plot...
Saving to fingerprint_viz_faerun_2048.html...


c:\Users\Happy\SRP\SRP Project MT training\mt_training\fingerprint_viz_faerun_2048.html.html

✓ Visualization saved: fingerprint_viz_faerun_2048.html


In [ ]:
# # Generate fingerprints at multiple folding lengths
# encoder = DrfpEncoder()
# fps_dict = {}  # Will store {folding_length: {dataset: fps}}

# for fold_len in folding_lengths:
#     print(f"\n=== Encoding with folding length {fold_len} ===")
#     fps_mit = encoder.encode(reaction_strings_list, show_progress_bar=True, n_folded_length=fold_len)
#     fps_chanlam = encoder.encode(reaction_strings_list_chanlam, show_progress_bar=True, n_folded_length=fold_len)
#     fps_dict[fold_len] = {"mit": fps_mit, "chanlam": fps_chanlam}
#     print(f"✓ Encoded USPTO-MIT: {len(fps_mit)} fingerprints")
#     print(f"✓ Encoded Chan–Lam: {len(fps_chanlam)} fingerprints")

# # Use 2048 as default for visualization
# fps = fps_dict[2048]["mit"]
# fps2 = fps_dict[2048]["chanlam"]
# print(f"\nDefault fingerprints (2048) sample: {fps[0][:20]}...")


In [ ]:
# # Compute and display metrics for each folding length
# print("\n" + "="*60)
# print("FINGERPRINT SPACE METRICS")
# print("="*60)

# metrics_list = []

# for fold_len in folding_lengths:
#     if fold_len not in fps_dict:
#         continue
    
#     fps_mit = np.array(fps_dict[fold_len]["mit"], dtype=bool)
#     fps_chanlam = np.array(fps_dict[fold_len]["chanlam"], dtype=bool)
#     fps_combined = np.vstack([fps_mit, fps_chanlam])
    
#     print(f"\nFolding Length: {fold_len}")
#     print(f"  MIT fingerprints: {len(fps_mit)}")
#     print(f"  Chan-Lam fingerprints: {len(fps_chanlam)}")
#     print(f"  Fingerprint dimensionality: {fps_mit.shape[1]}")
    
#     # Compute basic statistics
#     mit_sparsity = 1.0 - (np.mean(fps_mit) * 100)
#     chanlam_sparsity = 1.0 - (np.mean(fps_chanlam) * 100)
    
#     print(f"  MIT sparsity: {mit_sparsity:.2f}%")
#     print(f"  Chan-Lam sparsity: {chanlam_sparsity:.2f}%")
    
#     # Sample distances to check distance distribution
#     sample_size = min(100, len(fps_combined))
#     sample_indices = np.random.choice(len(fps_combined), sample_size, replace=False)
#     sample_fps = fps_combined[sample_indices]
#     sample_distances = compute_distances(sample_fps)
    
#     mean_distance = np.mean(sample_distances[np.triu_indices_from(sample_distances, k=1)])
#     min_distance = np.min(sample_distances[np.triu_indices_from(sample_distances, k=1)])
#     max_distance = np.max(sample_distances[np.triu_indices_from(sample_distances, k=1)])
    
#     print(f"  Distance stats (sample of {sample_size}):")
#     print(f"    Mean: {mean_distance:.4f}")
#     print(f"    Min:  {min_distance:.4f}")
#     print(f"    Max:  {max_distance:.4f}")
    
#     metrics_list.append({
#         "Folding Length": fold_len,
#         "MIT Samples": len(fps_mit),
#         "Chan-Lam Samples": len(fps_chanlam),
#         "Dimensionality": fps_mit.shape[1],
#         "MIT Sparsity (%)": f"{mit_sparsity:.2f}",
#         "Chan-Lam Sparsity (%)": f"{chanlam_sparsity:.2f}",
#         "Mean Distance": f"{mean_distance:.4f}"
#     })

# metrics_df = pd.DataFrame(metrics_list)
# print("\n" + "="*60)
# print("SUMMARY TABLE")
# print("="*60)
# print(metrics_df.to_string(index=False))


In [ ]:
# Check generated visualization files
import glob

print("\n" + "="*60)
print("GENERATED VISUALIZATIONS")
print("="*60)

html_files = glob.glob("fingerprint_viz_faerun_*.html")
if html_files:
    for file in sorted(html_files):
        file_size = os.path.getsize(file) / 1024  # KB
        print(f"✓ {file} ({file_size:.1f} KB)")
    print(f"\nOpen these HTML files in a browser to explore the interactive Faerun visualizations.")
else:
    print("No visualizations generated yet. Run the previous cells to generate them.")
